# Parte 3 - Recommendations Engine

## Objetivo

Desenvolver um sistema de recomendações capaz de analisar uma campanha publicitária e identificar oportunidades de melhoria nas características do criativo.

O sistema utilizará os resultados da análise exploratória e da modelagem preditiva para produzir recomendações específicas, quantificadas, contextualizadas e priorizadas.

### Estratégia

Será implementado um sistema híbrido que combina:

1. Previsões do modelo Ridge Regression;
2. Comparações descritivas entre campanhas semelhantes;
3. Regras para identificar características modificáveis;
4. Priorização por diferença estimada no Klike Score.

As recomendações representam hipóteses de otimização fundamentadas nos dados e não garantias de melhoria causal.


In [151]:
import pandas as pd
import numpy as np

import joblib

from pathlib import Path

from sklearn.base import clone

from IPython.display import display


In [152]:
# Identificar a raiz do projeto
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Caminho do dataset
DATA_PATH = (
    PROJECT_ROOT / "data" / "klike_challenge_dataset.csv"
)

print("Dataset encontrado:", DATA_PATH.exists())


Dataset encontrado: True


## 1 - Input e geração de recomendações

O Recommendations Engine receberá como entrada os atributos de uma campanha publicitária.

Cada campanha será representada por uma linha do dataset, contendo informações sobre plataforma, categoria, objetivo, público-alvo e características do vídeo.

A partir dessa entrada, o sistema avaliará possíveis alterações nas características modificáveis do criativo.

As recomendações serão geradas considerando a configuração atual da campanha e alternativas disponíveis para cada atributo.


In [153]:
# Carregar o dataset original
df = pd.read_csv(
    DATA_PATH,
    parse_dates=["date"]
)

# Verificar dimensões
print("Quantidade de campanhas:", df.shape[0])
print("Quantidade de variáveis:", df.shape[1])

display(df.head())


Quantidade de campanhas: 500
Quantidade de variáveis: 26


,campaign_id,date,platform,category,objective,target_audience_age,is_retargeting,video_duration_s,format,has_subtitle,...,clicks,ctr,cpc,spend,conversions,revenue,roas,avg_watch_time_s,engagement_rate,klike_score
0,KLK-0001,2024-01-01,Meta,Branding,awareness,18-24,False,NaN,vertical,True,...,5882,0.0390,0.82,4823.24,48,0.00,0.00,3.9,0.0446,63.6
1,KLK-0002,2024-03-22,LinkedIn,App Install,traffic,25-34,False,NaN,horizontal,True,...,493,0.0664,3.54,1745.22,34,NaN,0.16,6.4,0.0217,70.0
2,KLK-0003,2024-05-04,TikTok,SaaS,traffic,25-34,False,8.0,vertical,True,...,15069,0.0780,0.38,5726.22,492,51053.44,8.92,8.0,0.1000,99.1
3,KLK-0004,2024-12-27,TikTok,Lead Gen,engagement,25-34,False,27.0,quadrado,False,...,601,0.0925,0.71,426.71,19,1113.78,2.61,20.3,0.0414,74.9
4,KLK-0005,2024-07-10,Meta,Branding,engagement,25-34,False,34.0,vertical,True,...,2355,0.0369,0.38,894.90,13,0.00,0.00,34.0,0.0184,77.9


### 1.1 - Características modificáveis do criativo

O sistema deverá diferenciar características que podem ser alteradas no vídeo de informações relacionadas ao contexto da campanha.

**Características modificáveis:**

- Presença de hook;
- Presença de CTA;
- Presença de rosto humano;
- Utilização de legendas;
- Densidade de texto;
- Formato do vídeo;
- Duração do vídeo.

**Características de contexto:**

- Plataforma;
- Categoria;
- Objetivo;
- Público-alvo;
- Retargeting.

As características de contexto serão utilizadas para orientar as recomendações, mas não serão modificadas automaticamente pelo sistema.


In [154]:
# Características modificáveis
FEATURES_MODIFICAVEIS = [
    "has_hook",
    "has_cta",
    "has_face",
    "has_subtitle",
    "text_density",
    "format",
    "video_duration_s"
]

# Características de contexto
FEATURES_CONTEXTO = [
    "platform",
    "category",
    "objective",
    "target_audience_age",
    "is_retargeting"
]

print("Características modificáveis:")
for feature in FEATURES_MODIFICAVEIS:
    print("-", feature)

print("\nCaracterísticas de contexto:")
for feature in FEATURES_CONTEXTO:
    print("-", feature)


Características modificáveis:
- has_hook
- has_cta
- has_face
- has_subtitle
- text_density
- format
- video_duration_s

Características de contexto:
- platform
- category
- objective
- target_audience_age
- is_retargeting


### 1.2 - Carregamento do modelo preditivo

O Recommendations Engine utilizará o pipeline Ridge Regression desenvolvido na Parte 2 para estimar o Klike Score de campanhas.

O modelo será carregado a partir do arquivo serializado, preservando as transformações utilizadas durante o treinamento.

Antes de gerar recomendações, será realizada uma previsão de teste para verificar a integração entre o dataset e o modelo.


In [155]:
# Caminho do modelo treinado
MODEL_PATH = (
    PROJECT_ROOT / "models" / "ridge_klike_score.joblib"
)

# Verificar se o arquivo existe
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Modelo não encontrado: {MODEL_PATH}"
    )

# Carregar o pipeline treinado
modelo = joblib.load(MODEL_PATH)

print("Modelo carregado com sucesso!")


Modelo carregado com sucesso!


In [156]:
# Selecionar a primeira campanha
campanha = df.iloc[0].copy()

# Exibir informações principais
colunas_exibicao = [
    "campaign_id",
    "platform",
    "category",
    "objective",
    "has_hook",
    "has_cta",
    "has_face",
    "has_subtitle",
    "text_density",
    "format",
    "video_duration_s"
]

display(campanha[colunas_exibicao])


campaign_id          KLK-0001
platform                 Meta
category             Branding
objective           awareness
has_hook                False
has_cta                  True
has_face                 True
has_subtitle             True
text_density           medium
format               vertical
video_duration_s          NaN
Name: 0, dtype: object

### 1.3 - Preparação da campanha para previsão

Antes de utilizar o modelo, os dados de entrada precisam apresentar a mesma estrutura utilizada durante o treinamento.

As variáveis booleanas serão convertidas para as categorias "Sim" e "Não", preservando valores ausentes.

A preparação será aplicada sobre uma cópia dos dados da campanha, evitando alterações no dataset original.


In [157]:
# Variáveis booleanas utilizadas na modelagem
FEATURES_BOOLEANAS = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

def preparar_campanha(campanha):

    # Converter uma linha em DataFrame
    dados = pd.DataFrame([campanha]).copy()

    # Converter booleanos para categorias textuais
    for coluna in FEATURES_BOOLEANAS:

        dados[coluna] = dados[coluna].map({
            True: "Sim",
            False: "Não",
            "Sim": "Sim",
            "Não": "Não"
        })

    return dados


In [158]:
# Preparar campanha
campanha_preparada = preparar_campanha(
    campanha
)

# Gerar previsão
score_previsto = modelo.predict(
    campanha_preparada
)[0]

# Comparar score real e previsto
print(
    "Klike Score real:",
    round(campanha["klike_score"], 2)
)

print(
    "Klike Score previsto:",
    round(score_previsto, 2)
)


Klike Score real: 63.6
Klike Score previsto: 63.68


### 1.4 - Identificação de oportunidades de melhoria

Após validar a integração com o modelo preditivo, o próximo passo consiste em identificar possíveis modificações nas características do criativo.

O sistema avaliará atributos acionáveis, como presença de hook, CTA, rosto humano, legendas, densidade de texto e formato.

Para cada característica, serão consideradas alternativas diferentes da configuração atual da campanha.

Nesta etapa, serão geradas apenas alterações candidatas. A estimativa de impacto será realizada posteriormente, utilizando o modelo preditivo e as evidências da análise exploratória.


In [159]:
# Alternativas acionáveis dos criativos (configuração única)
ALTERNATIVAS = {
    "has_hook": ["Sim", "Não"],
    "has_cta": ["Sim", "Não"],
    "has_face": ["Sim", "Não"],
    "has_subtitle": ["Sim", "Não"],
    "text_density": ["low", "medium", "high"],
    "format": ["vertical", "horizontal", "quadrado"]
}

print("Alternativas configuradas:", ALTERNATIVAS)

Alternativas configuradas: {'has_hook': ['Sim', 'Não'], 'has_cta': ['Sim', 'Não'], 'has_face': ['Sim', 'Não'], 'has_subtitle': ['Sim', 'Não'], 'text_density': ['low', 'medium', 'high'], 'format': ['vertical', 'horizontal', 'quadrado']}


In [160]:
def gerar_alteracoes_candidatas(campanha_preparada):

    alteracoes = []

    # Percorrer características modificáveis
    for atributo, valores in ALTERNATIVAS.items():

        # Valor atual da campanha
        valor_atual = campanha_preparada.iloc[0][atributo]

        # Avaliar alternativas
        for novo_valor in valores:

            # Ignorar alterações sem mudança
            if valor_atual == novo_valor:
                continue

            # Criar uma alternativa
            alteracoes.append({
                "atributo": atributo,
                "valor_atual": valor_atual,
                "novo_valor": novo_valor
            })

    return pd.DataFrame(alteracoes)


In [161]:
# Gerar alternativas para a campanha selecionada
alteracoes_candidatas = gerar_alteracoes_candidatas(
    campanha_preparada
)

# Exibir alternativas
display(alteracoes_candidatas)


,atributo,valor_atual,novo_valor
0,has_hook,Não,Sim
1,has_cta,Sim,Não
2,has_face,Sim,Não
3,has_subtitle,Sim,Não
4,text_density,medium,low
5,text_density,medium,high
6,format,vertical,horizontal
7,format,vertical,quadrado


## 2 - Recomendações quantificadas

Nesta etapa, serão estimadas as diferenças no Klike Score previstas pelo modelo Ridge Regression para cada alteração candidata.

Para cada alternativa, será criada uma cópia da campanha original, modificando apenas uma característica do criativo.

O modelo será utilizado para estimar o score da configuração alternativa, permitindo calcular a diferença em relação à previsão original.

A diferença será expressa em pontos do Klike Score.

**Limitação metodológica:** as diferenças representam variações nas previsões do modelo, não efeitos causais comprovados. Portanto, não devem ser interpretadas como ganhos garantidos de performance.


In [162]:
def estimar_alteracoes(
    campanha_preparada,
    alteracoes_candidatas,
    modelo
):
    # Previsão da campanha original
    score_original = modelo.predict(
        campanha_preparada
    )[0]

    resultados = []

    # Avaliar cada alteração candidata
    for _, alteracao in alteracoes_candidatas.iterrows():

        atributo = alteracao["atributo"]
        novo_valor = alteracao["novo_valor"]

        # Criar cópia da campanha
        campanha_alternativa = campanha_preparada.copy()

        # Modificar apenas uma característica
        campanha_alternativa.loc[
            campanha_alternativa.index[0],
            atributo
        ] = novo_valor

        # Prever o score da alternativa
        score_alternativo = modelo.predict(
            campanha_alternativa
        )[0]

        # Calcular diferença prevista
        diferenca_score = (
            score_alternativo - score_original
        )

        # Armazenar resultado
        resultados.append({
            "atributo": atributo,
            "valor_atual": alteracao["valor_atual"],
            "novo_valor": novo_valor,
            "score_original": score_original,
            "score_alternativo": score_alternativo,
            "diferenca_score": diferenca_score
        })

    return pd.DataFrame(resultados)


In [163]:
# Estimar as alterações candidatas
estimativas = estimar_alteracoes(
    campanha_preparada=campanha_preparada,
    alteracoes_candidatas=alteracoes_candidatas,
    modelo=modelo
)

# Ordenar pela maior diferença prevista
estimativas = estimativas.sort_values(
    by="diferenca_score",
    ascending=False
).reset_index(drop=True)

# Exibir resultados
display(estimativas.round(2))


,atributo,valor_atual,novo_valor,score_original,score_alternativo,diferenca_score
0,has_hook,Não,Sim,63.68,81.70,18.02
1,format,vertical,quadrado,63.68,62.26,-1.42
2,format,vertical,horizontal,63.68,61.88,-1.79
3,text_density,medium,low,63.68,57.06,-6.62
4,has_cta,Sim,Não,63.68,56.27,-7.41
5,has_subtitle,Sim,Não,63.68,55.63,-8.04
6,has_face,Sim,Não,63.68,51.62,-12.06
7,text_density,medium,high,63.68,49.67,-14.01


In [164]:
# Selecionar alternativas com ganho previsto positivo
alternativas_favoraveis = estimativas[
    estimativas["diferenca_score"] > 0
].copy()

# Contar alternativas favoráveis
print(
    "Quantidade de alternativas favoráveis:",
    len(alternativas_favoraveis)
)

display(alternativas_favoraveis.round(2))


Quantidade de alternativas favoráveis: 1


,atributo,valor_atual,novo_valor,score_original,score_alternativo,diferenca_score
0,has_hook,Não,Sim,63.68,81.7,18.02


## 3 - Recomendações contextuais

Nesta etapa, as alterações candidatas serão complementadas com evidências de campanhas semelhantes.

A análise considerará inicialmente a plataforma e a categoria do anunciante, utilizando CTR e ROAS como métricas de performance.

O objetivo é verificar se as diferenças observadas no conjunto geral também aparecem em grupos comparáveis à campanha analisada.

As estimativas do modelo e as diferenças históricas serão apresentadas separadamente, pois representam medidas distintas.

Quando a quantidade de campanhas comparáveis for insuficiente, essa limitação será explicitada.


In [165]:
# Identificar plataforma e categoria
plataforma = campanha["platform"]
categoria = campanha["category"]

print("Plataforma:", plataforma)
print("Categoria:", categoria)

# Selecionar campanhas do mesmo contexto
campanhas_contexto = df[
    (df["platform"] == plataforma) &
    (df["category"] == categoria)
].copy()

print(
    "Campanhas no mesmo contexto:",
    len(campanhas_contexto)
)


Plataforma: Meta
Categoria: Branding
Campanhas no mesmo contexto: 30


In [166]:
def consultar_evidencias(
    dados,
    atributo,
    valor_atual,
    novo_valor
):
    # Selecionar campanhas com a configuração atual
    grupo_atual = dados[
        dados[atributo] == valor_atual
    ]

    # Selecionar campanhas com a configuração alternativa
    grupo_alternativo = dados[
        dados[atributo] == novo_valor
    ]

    # Verificar disponibilidade de dados
    if grupo_atual.empty or grupo_alternativo.empty:
        return {
            "n_atual": len(grupo_atual),
            "n_alternativo": len(grupo_alternativo),
            "diferenca_ctr_pct": np.nan,
            "diferenca_roas_pct": np.nan
        }

    # Calcular médias
    ctr_atual = grupo_atual["ctr"].mean()
    ctr_alternativo = grupo_alternativo["ctr"].mean()

    roas_atual = grupo_atual["roas"].mean()
    roas_alternativo = grupo_alternativo["roas"].mean()

    # Calcular diferenças relativas
    diferenca_ctr = np.nan
    diferenca_roas = np.nan

    if pd.notna(ctr_atual) and ctr_atual > 0:
        diferenca_ctr = (
            (ctr_alternativo - ctr_atual) /
            ctr_atual
        ) * 100

    if pd.notna(roas_atual) and roas_atual > 0:
        diferenca_roas = (
            (roas_alternativo - roas_atual) /
            roas_atual
        ) * 100

    return {
        "n_atual": len(grupo_atual),
        "n_alternativo": len(grupo_alternativo),
        "diferenca_ctr_pct": diferenca_ctr,
        "diferenca_roas_pct": diferenca_roas
    }


In [167]:
# Criar cópia dos dados históricos
df_historico = df.copy()

# Converter booleanos para o mesmo formato do modelo
for coluna in FEATURES_BOOLEANAS:

    df_historico[coluna] = df_historico[coluna].map({
        True: "Sim",
        False: "Não"
    })

print("Dados históricos preparados.")


Dados históricos preparados.


In [168]:
# Selecionar campanhas do mesmo contexto
dados_contexto = df_historico[
    (df_historico["platform"] == plataforma) &
    (df_historico["category"] == categoria)
].copy()

resultados_contextuais = []

for _, alteracao in estimativas.iterrows():

    evidencias = consultar_evidencias(
        dados=dados_contexto,
        atributo=alteracao["atributo"],
        valor_atual=alteracao["valor_atual"],
        novo_valor=alteracao["novo_valor"]
    )

    resultados_contextuais.append({
        "atributo": alteracao["atributo"],
        "valor_atual": alteracao["valor_atual"],
        "novo_valor": alteracao["novo_valor"],
        "diferenca_score": alteracao["diferenca_score"],
        **evidencias
    })

# Organizar resultados
tabela_contextual = pd.DataFrame(
    resultados_contextuais
)

display(tabela_contextual.round(2))


,atributo,valor_atual,novo_valor,diferenca_score,n_atual,n_alternativo,diferenca_ctr_pct,diferenca_roas_pct
0,has_hook,Não,Sim,18.02,13,17,30.76,NaN
1,format,vertical,quadrado,-1.42,18,6,41.09,NaN
2,format,vertical,horizontal,-1.79,18,6,-10.35,NaN
3,text_density,medium,low,-6.62,13,16,-23.05,NaN
4,has_cta,Sim,Não,-7.41,22,8,13.31,NaN
5,has_subtitle,Sim,Não,-8.04,16,9,-17.27,NaN
6,has_face,Sim,Não,-12.06,17,13,-2.34,NaN
7,text_density,medium,high,-14.01,13,1,-84.82,NaN


### 3.1 - Critério de evidência contextual

As comparações entre campanhas semelhantes podem apresentar instabilidade quando os grupos possuem poucas observações.

Para reduzir o risco de recomendações fundamentadas em amostras muito pequenas, será adotado um critério mínimo de 10 campanhas por grupo.

As evidências serão classificadas como:

- **Amostra suficiente para análise exploratória:** pelo menos 10 campanhas em ambos os grupos.
- **Amostra limitada:** menos de 10 campanhas em qualquer um dos grupos.
- **Sem evidência comparativa:** ausência de campanhas em pelo menos um dos grupos.

Esse critério é heurístico e não representa um teste de significância estatística.

As recomendações também deverão considerar a direção das diferenças observadas e possíveis divergências entre o modelo preditivo e as métricas históricas.


In [169]:
def classificar_evidencia(linha):

    n_atual = linha["n_atual"]
    n_alternativo = linha["n_alternativo"]

    if n_atual == 0 or n_alternativo == 0:
        return "Sem evidência comparativa"

    if min(n_atual, n_alternativo) < 10:
        return "Amostra limitada"

    return "Amostra suficiente para análise exploratória"


# Classificar cada alternativa
tabela_contextual["qualidade_evidencia"] = (
    tabela_contextual.apply(
        classificar_evidencia,
        axis=1
    )
)

display(tabela_contextual.round(2))


,atributo,valor_atual,novo_valor,diferenca_score,n_atual,n_alternativo,diferenca_ctr_pct,diferenca_roas_pct,qualidade_evidencia
0,has_hook,Não,Sim,18.02,13,17,30.76,NaN,Amostra suficiente para análise exploratória
1,format,vertical,quadrado,-1.42,18,6,41.09,NaN,Amostra limitada
2,format,vertical,horizontal,-1.79,18,6,-10.35,NaN,Amostra limitada
3,text_density,medium,low,-6.62,13,16,-23.05,NaN,Amostra suficiente para análise exploratória
4,has_cta,Sim,Não,-7.41,22,8,13.31,NaN,Amostra limitada
5,has_subtitle,Sim,Não,-8.04,16,9,-17.27,NaN,Amostra limitada
6,has_face,Sim,Não,-12.06,17,13,-2.34,NaN,Amostra suficiente para análise exploratória
7,text_density,medium,high,-14.01,13,1,-84.82,NaN,Amostra limitada



### 3.2 - Interpretação das evidências contextuais

A análise contextual considerou campanhas da mesma plataforma e categoria do criativo selecionado.

Foram identificadas 30 campanhas comparáveis, todas com ROAS igual a zero. Dessa forma, não foi possível calcular diferenças percentuais de ROAS entre as alternativas.

Essa limitação foi preservada na análise, sem substituir os resultados indefinidos por valores artificiais.

**Principais resultados**

A inclusão de hook apresentou diferença prevista de +18,02 pontos no Klike Score. Nas campanhas comparáveis, a presença de hook também esteve associada a CTR médio 30,76% superior.

A alteração do formato vertical para quadrado apresentou CTR histórico 41,09% superior, mas diferença prevista de -1,42 ponto no score. Além disso, o grupo alternativo contém apenas seis campanhas.

As alterações de densidade de texto não apresentaram evidências favoráveis no contexto analisado.

**Conclusão**

A inclusão de hook foi identificada como a principal oportunidade de melhoria para a campanha.

As demais alternativas apresentam resultados conflitantes, negativos ou sustentados por amostras limitadas.

Essas diferenças são associações observacionais e não representam efeitos causais garantidos.


### 3.3 -Estratégia de segmentação hierárquica

Para tornar as recomendações mais relevantes, será utilizada uma estratégia de comparação hierárquica entre campanhas.

A análise priorizará campanhas com a mesma plataforma, categoria e objetivo.

Quando a quantidade de observações for insuficiente, o contexto será ampliado progressivamente:

1. Plataforma, categoria e objetivo;
2. Plataforma e objetivo;
3. Plataforma.

Será adotado um mínimo heurístico de 10 campanhas em cada grupo comparado.

A ampliação do contexto permite obter mais observações, mas reduz a similaridade entre as campanhas.

Por isso, o nível de segmentação utilizado será informado junto às evidências.

As comparações continuarão sendo observacionais, não representando estimativas causais.


In [170]:
def selecionar_contexto(
    dados,
    campanha,
    atributo,
    valor_atual,
    novo_valor,
    minimo_amostras=10
):

    niveis = [
        ["platform", "category", "objective"],
        ["platform", "objective"],
        ["platform"]
    ]

    ultimo_contexto = None
    ultimo_nivel = None

    for colunas in niveis:

        # Filtrar campanhas pelo contexto
        contexto = dados.copy()

        for coluna in colunas:
            contexto = contexto[
                contexto[coluna] == campanha[coluna]
            ]

        # Contar campanhas por configuração
        n_atual = (
            contexto[atributo] == valor_atual
        ).sum()

        n_alternativo = (
            contexto[atributo] == novo_valor
        ).sum()

        ultimo_contexto = contexto
        ultimo_nivel = " + ".join(colunas)

        # Verificar quantidade mínima
        if (
            n_atual >= minimo_amostras
            and n_alternativo >= minimo_amostras
        ):
            return contexto, ultimo_nivel

    # Caso nenhum nível atenda ao critério
    return ultimo_contexto, ultimo_nivel


In [171]:
resultados_hierarquicos = []

for _, alteracao in estimativas.iterrows():

    # Selecionar contexto adequado
    contexto, nivel = selecionar_contexto(
        dados=df_historico,
        campanha=campanha,
        atributo=alteracao["atributo"],
        valor_atual=alteracao["valor_atual"],
        novo_valor=alteracao["novo_valor"]
    )

    # Calcular evidências históricas
    evidencias = consultar_evidencias(
        dados=contexto,
        atributo=alteracao["atributo"],
        valor_atual=alteracao["valor_atual"],
        novo_valor=alteracao["novo_valor"]
    )

    resultados_hierarquicos.append({
        "atributo": alteracao["atributo"],
        "valor_atual": alteracao["valor_atual"],
        "novo_valor": alteracao["novo_valor"],
        "diferenca_score": alteracao["diferenca_score"],
        "nivel_contexto": nivel,
        **evidencias
    })

# Criar tabela
tabela_hierarquica = pd.DataFrame(
    resultados_hierarquicos
)

# Classificar evidência
tabela_hierarquica["qualidade_evidencia"] = (
    tabela_hierarquica.apply(
        classificar_evidencia,
        axis=1
    )
)

display(tabela_hierarquica.round(2))


,atributo,valor_atual,novo_valor,diferenca_score,nivel_contexto,n_atual,n_alternativo,diferenca_ctr_pct,diferenca_roas_pct,qualidade_evidencia
0,has_hook,Não,Sim,18.02,platform,113,128,34.31,-23.15,Amostra suficiente para análise exploratória
1,format,vertical,quadrado,-1.42,platform + objective,13,10,54.46,173.16,Amostra suficiente para análise exploratória
2,format,vertical,horizontal,-1.79,platform,106,67,10.95,-9.02,Amostra suficiente para análise exploratória
3,text_density,medium,low,-6.62,platform,105,104,-1.78,19.22,Amostra suficiente para análise exploratória
4,has_cta,Sim,Não,-7.41,platform,174,67,-5.42,-7.88,Amostra suficiente para análise exploratória
5,has_subtitle,Sim,Não,-8.04,platform,128,92,-11.56,5.57,Amostra suficiente para análise exploratória
6,has_face,Sim,Não,-12.06,platform,123,118,-10.23,15.81,Amostra suficiente para análise exploratória
7,text_density,medium,high,-14.01,platform,105,32,-34.46,-11.93,Amostra suficiente para análise exploratória


## 4. - Priorização das recomendações

Nesta etapa, as alternativas serão classificadas e ordenadas conforme as evidências disponíveis.

A priorização considerará:

1. Diferença prevista no Klike Score;
2. Direção da diferença histórica na métrica contextual;
3. Quantidade de campanhas comparáveis;
4. Nível de segmentação utilizado.

As alternativas serão classificadas como:

- **Melhoria prevista:** diferença positiva no score e na métrica contextual.
- **Hipótese para teste:** evidência contextual positiva, mas diferença não positiva no score.
- **Não priorizada:** ausência de evidências favoráveis suficientes.

A classificação não representa uma estimativa causal e não garante que as alterações produzirão os resultados observados.


In [172]:
def classificar_alternativa(linha):

    diferenca_score = linha["diferenca_score"]
    diferenca_ctr = linha["diferenca_ctr_pct"]

    # Verificar qualidade mínima da evidência
    evidencia_suficiente = (
        linha["qualidade_evidencia"]
        == "Amostra suficiente para análise exploratória"
    )

    # Melhoria prevista pelo modelo e evidência histórica
    if (
        diferenca_score > 0
        and pd.notna(diferenca_ctr)
        and diferenca_ctr > 0
        and evidencia_suficiente
    ):
        return "Melhoria prevista"

    # Evidência histórica favorável, mas modelo divergente
    if (
        pd.notna(diferenca_ctr)
        and diferenca_ctr > 0
        and evidencia_suficiente
    ):
        return "Hipótese para teste"

    return "Não priorizada"


# Criar cópia da tabela
tabela_priorizacao = tabela_hierarquica.copy()

# Classificar alternativas
tabela_priorizacao["classificacao"] = (
    tabela_priorizacao.apply(
        classificar_alternativa,
        axis=1
    )
)

display(tabela_priorizacao.round(2))


,atributo,valor_atual,novo_valor,diferenca_score,nivel_contexto,n_atual,n_alternativo,diferenca_ctr_pct,diferenca_roas_pct,qualidade_evidencia,classificacao
0,has_hook,Não,Sim,18.02,platform,113,128,34.31,-23.15,Amostra suficiente para análise exploratória,Melhoria prevista
1,format,vertical,quadrado,-1.42,platform + objective,13,10,54.46,173.16,Amostra suficiente para análise exploratória,Hipótese para teste
2,format,vertical,horizontal,-1.79,platform,106,67,10.95,-9.02,Amostra suficiente para análise exploratória,Hipótese para teste
3,text_density,medium,low,-6.62,platform,105,104,-1.78,19.22,Amostra suficiente para análise exploratória,Não priorizada
4,has_cta,Sim,Não,-7.41,platform,174,67,-5.42,-7.88,Amostra suficiente para análise exploratória,Não priorizada
5,has_subtitle,Sim,Não,-8.04,platform,128,92,-11.56,5.57,Amostra suficiente para análise exploratória,Não priorizada
6,has_face,Sim,Não,-12.06,platform,123,118,-10.23,15.81,Amostra suficiente para análise exploratória,Não priorizada
7,text_density,medium,high,-14.01,platform,105,32,-34.46,-11.93,Amostra suficiente para análise exploratória,Não priorizada


In [173]:
# Definir prioridade das classificações
ordem_prioridade = {
    "Melhoria prevista": 0,
    "Hipótese para teste": 1,
    "Não priorizada": 2
}

# Criar coluna auxiliar
tabela_priorizacao["ordem_prioridade"] = (
    tabela_priorizacao["classificacao"].map(
        ordem_prioridade
    )
)

# Ordenar alternativas
tabela_priorizacao = tabela_priorizacao.sort_values(
    by=[
        "ordem_prioridade",
        "diferenca_score",
        "diferenca_ctr_pct"
    ],
    ascending=[True, False, False]
).reset_index(drop=True)

display(
    tabela_priorizacao[
        [
            "atributo",
            "valor_atual",
            "novo_valor",
            "diferenca_score",
            "diferenca_ctr_pct",
            "nivel_contexto",
            "classificacao"
        ]
    ].round(2)
)


,atributo,valor_atual,novo_valor,diferenca_score,diferenca_ctr_pct,nivel_contexto,classificacao
0,has_hook,Não,Sim,18.02,34.31,platform,Melhoria prevista
1,format,vertical,quadrado,-1.42,54.46,platform + objective,Hipótese para teste
2,format,vertical,horizontal,-1.79,10.95,platform,Hipótese para teste
3,text_density,medium,low,-6.62,-1.78,platform,Não priorizada
4,has_cta,Sim,Não,-7.41,-5.42,platform,Não priorizada
5,has_subtitle,Sim,Não,-8.04,-11.56,platform,Não priorizada
6,has_face,Sim,Não,-12.06,-10.23,platform,Não priorizada
7,text_density,medium,high,-14.01,-34.46,platform,Não priorizada


## 5. - Protótipo funcional

Nesta etapa, será implementada uma função que integra as etapas anteriores do Recommendations Engine.

O protótipo receberá uma campanha e executará automaticamente:

1. Preparação das características de entrada;
2. Identificação de alterações candidatas;
3. Estimativa de diferenças no Klike Score;
4. Consulta de evidências históricas contextualizadas;
5. Classificação e priorização das alternativas.

O resultado será apresentado em formato tabular, contendo recomendações específicas e suas respectivas evidências quantitativas.

As diferenças previstas e históricas serão identificadas separadamente, evitando interpretar associações como efeitos causais.


In [174]:
def recomendar_melhorias(campanha, dados_historicos, modelo):

    # Excluir a campanha analisada do histórico
    historico_comparavel = dados_historicos[
        dados_historicos["campaign_id"] != campanha["campaign_id"]
    ].copy()

    # 1. Preparar a campanha
    campanha_preparada = preparar_campanha(campanha)

    # 2. Gerar alterações candidatas
    candidatas = gerar_alteracoes_candidatas(
        campanha_preparada
    )

    # 3. Estimar diferenças no score
    estimativas = estimar_alteracoes(
        campanha_preparada,
        candidatas,
        modelo
    )

    resultados = []

    # 4. Consultar evidências contextuais
    for _, alteracao in estimativas.iterrows():

        contexto, nivel = selecionar_contexto(
            dados=historico_comparavel,
            campanha=campanha,
            atributo=alteracao["atributo"],
            valor_atual=alteracao["valor_atual"],
            novo_valor=alteracao["novo_valor"]
        )

        evidencias = consultar_evidencias(
            dados=contexto,
            atributo=alteracao["atributo"],
            valor_atual=alteracao["valor_atual"],
            novo_valor=alteracao["novo_valor"]
        )

        resultado = {
            "atributo": alteracao["atributo"],
            "valor_atual": alteracao["valor_atual"],
            "novo_valor": alteracao["novo_valor"],
            "diferenca_score": alteracao["diferenca_score"],
            "nivel_contexto": nivel,
            **evidencias
        }

        resultado["qualidade_evidencia"] = (
            classificar_evidencia(resultado)
        )

        resultado["classificacao"] = (
            classificar_alternativa(resultado)
        )

        resultados.append(resultado)

    # 5. Criar tabela
    recomendacoes = pd.DataFrame(resultados)

    # Tratar caso sem alternativas
    if recomendacoes.empty:
        return recomendacoes

    # 6. Definir prioridade
    ordem_prioridade = {
        "Melhoria prevista": 0,
        "Hipótese para teste": 1,
        "Não priorizada": 2
    }

    recomendacoes["ordem_prioridade"] = (
        recomendacoes["classificacao"].map(
            ordem_prioridade
        )
    )

    # 7. Ordenar
    recomendacoes = recomendacoes.sort_values(
        by=[
            "ordem_prioridade",
            "diferenca_score",
            "diferenca_ctr_pct"
        ],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    return recomendacoes


In [175]:
# Selecionar uma campanha
campanha_teste = df.iloc[0].copy()

# Gerar recomendações
recomendacoes_teste = recomendar_melhorias(
    campanha=campanha_teste,
    dados_historicos=df_historico,
    modelo=modelo
)

# Exibir as recomendações
display(
    recomendacoes_teste[
        [
            "atributo",
            "valor_atual",
            "novo_valor",
            "diferenca_score",
            "diferenca_ctr_pct",
            "nivel_contexto",
            "classificacao"
        ]
    ].round(2)
)


,atributo,valor_atual,novo_valor,diferenca_score,diferenca_ctr_pct,nivel_contexto,classificacao
0,has_hook,Não,Sim,18.02,33.87,platform,Melhoria prevista
1,format,vertical,quadrado,-1.42,51.60,platform + objective,Hipótese para teste
2,format,vertical,horizontal,-1.79,10.49,platform,Hipótese para teste
3,text_density,medium,low,-6.62,-2.24,platform,Não priorizada
4,has_cta,Sim,Não,-7.41,-5.67,platform,Não priorizada
5,has_subtitle,Sim,Não,-8.04,-11.89,platform,Não priorizada
6,has_face,Sim,Não,-12.06,-10.58,platform,Não priorizada
7,text_density,medium,high,-14.01,-34.77,platform,Não priorizada


### 5.2 - Apresentação das recomendações

O protótipo será complementado com uma camada de comunicação voltada a profissionais de marketing.

As recomendações serão apresentadas em linguagem natural, destacando:

- A característica que poderá ser modificada;
- A alteração sugerida;
- A diferença prevista no Klike Score;
- A diferença histórica observada no CTR;
- O contexto utilizado para a comparação;
- As limitações da evidência.

As alternativas serão diferenciadas entre melhorias previstas, hipóteses para teste e alterações não priorizadas.

Essa abordagem busca transformar resultados técnicos em informações acionáveis, sem apresentar associações observacionais como efeitos causais garantidos.


In [176]:
# Nomes amigáveis para os atributos
NOMES_ATRIBUTOS = {
    "has_hook": "hook nos primeiros segundos",
    "has_cta": "call-to-action",
    "has_face": "presença de rosto humano",
    "has_subtitle": "legendas",
    "text_density": "densidade de texto",
    "format": "formato do vídeo"
}



def gerar_mensagem_recomendacao(linha):

    atributo = linha["atributo"]
    valor_atual = linha["valor_atual"]
    novo_valor = linha["novo_valor"]

    diferenca_score = linha["diferenca_score"]
    diferenca_ctr = linha["diferenca_ctr_pct"]

    classificacao = linha["classificacao"]

    n_atual = linha["n_atual"]
    n_alternativo = linha["n_alternativo"]

    contexto = linha["nivel_contexto"]

    nome_atributo = NOMES_ATRIBUTOS.get(
        atributo,
        atributo
    )

    # Descrever a ação
    if atributo.startswith("has_"):

        if novo_valor == "Sim":
            acao = f"Testar a inclusão de {nome_atributo}"

        elif novo_valor == "Não":
            acao = f"Testar a remoção de {nome_atributo}"

        else:
            acao = f"Verificar a configuração de {nome_atributo}"

    else:
        acao = (
            f"Testar a mudança de {nome_atributo} "
            f"de '{valor_atual}' para '{novo_valor}'"
        )


    # Descrever a previsão
    texto_score = (
        f"O modelo estima uma diferença de "
        f"{diferenca_score:+.2f} pontos no Klike Score."
    )

    # Descrever a evidência histórica
    if pd.notna(diferenca_ctr):
        texto_historico = (
            f"Historicamente, a alternativa apresentou "
            f"CTR médio {abs(diferenca_ctr):.1f}% "
            f"{'superior' if diferenca_ctr > 0 else 'inferior'} "
            f"à configuração atual."
        )
    else:
        texto_historico = (
            "Não foi possível estimar uma diferença "
            "histórica de CTR."
        )

    # Descrever a amostra
    texto_amostra = (
        f"A comparação considera {n_atual} campanhas "
        f"na configuração atual e {n_alternativo} "
        f"na alternativa, utilizando o contexto: {contexto}."
    )

    # Explicar a classificação
    if classificacao == "Melhoria prevista":
        texto_decisao = (
            "O modelo e a evidência histórica apontam "
            "uma direção favorável. Recomenda-se "
            "validar a alteração por experimento."
        )

    elif classificacao == "Hipótese para teste":
        texto_decisao = (
            "A evidência histórica é favorável, mas "
            "o modelo não prevê aumento do score. "
            "A alteração deve ser tratada como hipótese "
            "exploratória para teste A/B."
        )

    else:
        texto_decisao = (
            "A alteração não apresenta evidências "
            "suficientemente favoráveis para priorização."
        )

    mensagem = (
        f"{acao}.\n"
        f"{texto_score}\n"
        f"{texto_historico}\n"
        f"{texto_amostra}\n"
        f"{texto_decisao}\n"
        "As diferenças são associativas e não "
        "representam ganhos garantidos."
    )

    return mensagem


In [177]:
# Selecionar melhorias e hipóteses para teste
recomendacoes_priorizadas = recomendacoes_teste[
    recomendacoes_teste["classificacao"].isin([
        "Melhoria prevista",
        "Hipótese para teste"
    ])
].copy()

# Gerar mensagens
recomendacoes_priorizadas["mensagem"] = (
    recomendacoes_priorizadas.apply(
        gerar_mensagem_recomendacao,
        axis=1
    )
)

# Exibir recomendações
for indice, (_, linha) in enumerate(
    recomendacoes_priorizadas.iterrows(),
    start=1
):
    print(f"\nRECOMENDAÇÃO {indice}")
    print("=" * 60)
    print(linha["mensagem"])



RECOMENDAÇÃO 1
Testar a inclusão de hook nos primeiros segundos.
O modelo estima uma diferença de +18.02 pontos no Klike Score.
Historicamente, a alternativa apresentou CTR médio 33.9% superior à configuração atual.
A comparação considera 112 campanhas na configuração atual e 128 na alternativa, utilizando o contexto: platform.
O modelo e a evidência histórica apontam uma direção favorável. Recomenda-se validar a alteração por experimento.
As diferenças são associativas e não representam ganhos garantidos.

RECOMENDAÇÃO 2
Testar a mudança de formato do vídeo de 'vertical' para 'quadrado'.
O modelo estima uma diferença de -1.42 pontos no Klike Score.
Historicamente, a alternativa apresentou CTR médio 51.6% superior à configuração atual.
A comparação considera 12 campanhas na configuração atual e 10 na alternativa, utilizando o contexto: platform + objective.
A evidência histórica é favorável, mas o modelo não prevê aumento do score. A alteração deve ser tratada como hipótese exploratór

### 5.3 - Validação em diferentes campanhas

Para verificar a capacidade de generalização funcional do Recommendations Engine, o protótipo será testado com campanhas de diferentes plataformas.

Serão selecionadas campanhas de Meta, TikTok e LinkedIn, permitindo verificar:

- Se a função executa corretamente para diferentes entradas;
- Se as recomendações variam conforme as características da campanha;
- Se a segmentação contextual é aplicada corretamente;
- Se existem campanhas com menos de três sugestões priorizadas;
- Se as recomendações apresentam informações quantitativas e limitações interpretáveis.

Essa validação verifica o funcionamento do protótipo, não a eficácia causal das recomendações.


In [178]:
# Selecionar uma campanha de cada plataforma
campanhas_validacao = (
    df.groupby("platform", sort=True)
    .head(1)
    .copy()
)

# Exibir campanhas selecionadas
display(
    campanhas_validacao[
        [
            "campaign_id",
            "platform",
            "category",
            "objective",
            "has_hook",
            "has_cta",
            "has_face",
            "text_density",
            "format"
        ]
    ]
)


,campaign_id,platform,category,objective,has_hook,has_cta,has_face,text_density,format
0,KLK-0001,Meta,Branding,awareness,False,True,True,medium,vertical
1,KLK-0002,LinkedIn,App Install,traffic,True,True,True,low,horizontal
2,KLK-0003,TikTok,SaaS,traffic,True,True,True,medium,vertical


In [179]:
# Armazenar resultados dos testes
resultados_testes = []

for _, campanha_validacao in campanhas_validacao.iterrows():

    # Executar Recommendations Engine
    recomendacoes = recomendar_melhorias(
        campanha=campanha_validacao,
        dados_historicos=df_historico,
        modelo=modelo
    )

    # Selecionar recomendações priorizadas
    priorizadas = recomendacoes[
        recomendacoes["classificacao"].isin([
            "Melhoria prevista",
            "Hipótese para teste"
        ])
    ]

    # Registrar resultados
    resultados_testes.append({
        "Plataforma": campanha_validacao["platform"],
        "Campanha": campanha_validacao["campaign_id"],
        "Alternativas avaliadas": len(recomendacoes),
        "Sugestões priorizadas": len(priorizadas),
        "Melhorias previstas": (
            priorizadas["classificacao"] == "Melhoria prevista"
        ).sum(),
        "Hipóteses para teste": (
            priorizadas["classificacao"] == "Hipótese para teste"
        ).sum()
    })

# Criar tabela de validação
tabela_testes = pd.DataFrame(resultados_testes)

display(tabela_testes)


,Plataforma,Campanha,Alternativas avaliadas,Sugestões priorizadas,Melhorias previstas,Hipóteses para teste
0,Meta,KLK-0001,8,3,1,2
1,LinkedIn,KLK-0002,8,3,2,1
2,TikTok,KLK-0003,8,3,0,3


### 5.4 - Ampliação das alternativas acionáveis e decisão sobre duração

A validação inicial mostrou que considerar apenas a inclusão de atributos binários limitava a cobertura. Por isso, a configuração única de `ALTERNATIVAS`, definida na seção 1.4, avalia tanto a inclusão quanto a remoção de hook, CTA, rosto humano e legendas, além de formatos e densidades de texto.

Também investigamos a duração do vídeo: as campanhas de exemplo Meta e LinkedIn tinham `video_duration_s` ausente, embora essa variável esteja disponível para a maioria das campanhas das três plataformas. Optamos por **não incorporar duração como alternativa automática**: faltam informações na campanha de entrada em alguns casos, a variável teve baixa importância preditiva na Parte 2 e comparações de durações exatas poderiam produzir grupos históricos muito pequenos.

A ampliação das alternativas não implica que todas sejam benéficas; o engine mantém separadas melhorias previstas, hipóteses exploratórias e alterações não priorizadas.

### 5.5 - Avaliação da cobertura do Recommendations Engine

Após validar o funcionamento do protótipo em campanhas de Meta, LinkedIn e TikTok, será realizada uma avaliação de cobertura em todo o dataset.

O objetivo é verificar:

- Quantas campanhas recebem pelo menos três sugestões priorizadas;
- Quantas recebem menos de três sugestões;
- A distribuição entre melhorias previstas e hipóteses para teste;
- Se existem diferenças de cobertura entre plataformas.

Essa análise avalia a capacidade funcional do sistema de produzir recomendações, não a eficácia das alterações sugeridas.


In [180]:
# Armazenar resultados de cobertura
resultados_cobertura = []

for _, campanha_atual in df.iterrows():

    recomendacoes = recomendar_melhorias(
        campanha=campanha_atual,
        dados_historicos=df_historico,
        modelo=modelo
    )

    # Selecionar sugestões priorizadas
    priorizadas = recomendacoes[
        recomendacoes["classificacao"].isin([
            "Melhoria prevista",
            "Hipótese para teste"
        ])
    ]

    resultados_cobertura.append({
        "campaign_id": campanha_atual["campaign_id"],
        "platform": campanha_atual["platform"],
        "alternativas": len(recomendacoes),
        "sugestoes": len(priorizadas),
        "melhorias": (
            priorizadas["classificacao"] == "Melhoria prevista"
        ).sum(),
        "hipoteses": (
            priorizadas["classificacao"] == "Hipótese para teste"
        ).sum()
    })

# Criar tabela
tabela_cobertura = pd.DataFrame(
    resultados_cobertura
)

display(tabela_cobertura.head())


,campaign_id,platform,alternativas,sugestoes,melhorias,hipoteses
0,KLK-0001,Meta,8,3,1,2
1,KLK-0002,LinkedIn,8,3,2,1
2,KLK-0003,TikTok,8,3,0,3
3,KLK-0004,TikTok,8,3,1,2
4,KLK-0005,Meta,8,3,1,2


In [181]:
# Calcular indicadores gerais
total_campanhas = len(tabela_cobertura)

campanhas_com_tres = (
    tabela_cobertura["sugestoes"] >= 3
).sum()

campanhas_insuficientes = (
    tabela_cobertura["sugestoes"] < 3
).sum()

percentual_cobertura = (
    campanhas_com_tres / total_campanhas
) * 100

print("Total de campanhas:", total_campanhas)
print("Campanhas com 3+ sugestões:", campanhas_com_tres)
print("Campanhas com menos de 3:", campanhas_insuficientes)
print(f"Cobertura: {percentual_cobertura:.2f}%")


Total de campanhas: 500
Campanhas com 3+ sugestões: 392
Campanhas com menos de 3: 108
Cobertura: 78.40%


In [182]:
# Calcular cobertura por plataforma
cobertura_plataforma = (
    tabela_cobertura
    .assign(
        atende_minimo=lambda dados: dados["sugestoes"] >= 3
    )
    .groupby("platform")
    .agg(
        campanhas=("campaign_id", "count"),
        cobertura_pct=("atende_minimo", "mean"),
        media_sugestoes=("sugestoes", "mean"),
        media_melhorias=("melhorias", "mean"),
        media_hipoteses=("hipoteses", "mean")
    )
)

# Converter cobertura para percentual
cobertura_plataforma["cobertura_pct"] *= 100

display(cobertura_plataforma.round(2))


,campanhas,cobertura_pct,media_sugestoes,media_melhorias,media_hipoteses
platform,,,,,
LinkedIn,83,69.88,3.28,2.23,1.05
Meta,241,77.18,3.60,2.27,1.32
TikTok,176,84.09,3.65,2.19,1.47


### 5.5.1 - Interpretação dos resultados de cobertura

O Recommendations Engine foi executado nas 500 campanhas do dataset para avaliar sua capacidade de produzir sugestões em diferentes contextos.

**Cobertura geral**

- Campanhas avaliadas: 500;
- Campanhas com pelo menos três sugestões priorizadas: 392;
- Campanhas com menos de três sugestões: 108;
- Cobertura geral: 78,4%.

**Cobertura por plataforma**

- LinkedIn: 69,88%;
- Meta: 77,18%;
- TikTok: 84,09%.

O TikTok apresentou a maior cobertura, enquanto o LinkedIn registrou a menor.

**Distribuição das sugestões**

- LinkedIn: média de 2,23 melhorias previstas e 1,05 hipóteses por campanha.
- Meta: média de 2,27 melhorias previstas e 1,32 hipóteses por campanha.
- TikTok: média de 2,19 melhorias previstas e 1,47 hipóteses por campanha.

**Interpretação**

O protótipo demonstrou capacidade de produzir recomendações em diferentes plataformas, mas não garantiu três sugestões priorizadas para todas as campanhas.

Das 108 campanhas com cobertura insuficiente, 76 receberam duas sugestões, 27 receberam uma e cinco não receberam nenhuma.

**Limitações**

- A cobertura mede a quantidade de sugestões priorizadas, não sua eficácia real.
- As diferenças históricas são observacionais e não demonstram causalidade.
- O CTR é utilizado como métrica contextual independentemente do objetivo da campanha.
- A campanha analisada é excluída do histórico, mas ainda não existe uma restrição temporal para utilizar apenas campanhas anteriores.
- O limite mínimo de dez campanhas por grupo é heurístico e não garante significância estatística.

**Conclusão**

O Recommendations Engine alcançou cobertura de 78,4% para o requisito de três sugestões priorizadas.

O resultado demonstra viabilidade funcional e identifica oportunidades de evolução relacionadas à cobertura, seleção de métricas e validação experimental das recomendações.


In [183]:
# Identificar campanhas com cobertura insuficiente
campanhas_insuficientes = tabela_cobertura[
    tabela_cobertura["sugestoes"] < 3
].copy()

# Verificar a distribuição das quantidades
distribuicao_sugestoes = (
    tabela_cobertura["sugestoes"]
    .value_counts()
    .sort_index()
)

print("Distribuição de sugestões por campanha:")
display(distribuicao_sugestoes)

print("\nExemplos de campanhas com menos de três sugestões:")
display(campanhas_insuficientes.head(10))


Distribuição de sugestões por campanha:


sugestoes
0      5
1     27
2     76
3    140
4    133
5     75
6     34
7      8
8      2
Name: count, dtype: int64


Exemplos de campanhas com menos de três sugestões:


,campaign_id,platform,alternativas,sugestoes,melhorias,hipoteses
5,KLK-0006,Meta,9,2,0,2
6,KLK-0007,Meta,8,2,1,1
12,KLK-0013,TikTok,8,1,1,0
30,KLK-0031,TikTok,8,2,1,1
34,KLK-0035,LinkedIn,8,2,2,0
43,KLK-0044,LinkedIn,9,0,0,0
44,KLK-0045,Meta,8,2,0,2
45,KLK-0046,TikTok,8,2,1,1
48,KLK-0049,TikTok,8,2,0,2
49,KLK-0050,Meta,9,2,1,1


### 5.6 - Limitações da seleção de métricas

O protótipo utiliza o CTR como métrica histórica complementar para classificar as alternativas.

Entretanto, diferentes objetivos de campanha podem exigir métricas distintas.

Por exemplo, campanhas de tráfego podem priorizar CTR, enquanto campanhas de conversão podem exigir indicadores relacionados a conversões ou retorno financeiro.

Para campanhas de awareness, CTR não representa diretamente reconhecimento de marca.

A definição de métricas específicas por objetivo depende da disponibilidade e da qualidade dos indicadores no dataset.

Por isso, a utilização de CTR como métrica universal será tratada como uma simplificação do protótipo e uma oportunidade de evolução futura.

### Conclusão e próximos passos

O protótipo recebe uma campanha, testa alterações isoladas, calcula diferenças **previstas** no Klike Score, consulta diferenças **históricas** de CTR em grupos comparáveis e prioriza sugestões de forma transparente. As duas medidas não são intercambiáveis e não devem ser apresentadas como efeitos causais.

Antes da utilização em produção, seria necessário validar recomendações por experimentos controlados, escolher métricas adequadas ao objetivo da campanha e restringir o histórico a observações disponíveis na data da recomendação.